# LambadaClips Cloud Endpoints (Google Colab)
Notebook ini akan menjalankan API server untuk model Open-Source (SDXL, CogVideoX, MuseTalk) yang akan dihubungkan ke aplikasi LambadaClips Anda.

**Langkah-langkah:**
1. Pastikan Anda sudah terhubung ke kernel **Google Colab**.
2. Ubah tipe *Runtime* ke **T4 GPU** (atau yang lebih tinggi) jika belum.
3. Masukkan `NGROK_AUTH_TOKEN` Anda pada sel di bawah.
4. Jalankan semua sel (`Run All`).
5. Salin URL publik Ngrok yang muncul ke UI LambadaClips Anda!

In [ ]:
# 1. Install Dependencies
!pip install -q diffusers transformers accelerate invisible_watermark safetensors fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# 2. Setup Server dan SDXL Model
import torch
from diffusers import StableDiffusionXLPipeline
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import base64
from io import BytesIO
from pyngrok import ngrok
import threading
import nest_asyncio

# Izinkan Uvicorn berjalan di dalam event loop Colab (Jupyter)
nest_asyncio.apply()

print("Loading SDXL Model... (Ini membutuhkan waktu beberapa menit)")
# Uncomment baris di bawah ini untuk mengunduh dan menjalankan model asli SDXL:
# pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")

app = FastAPI(title="LambadaClips Cloud API")

class GenerateRequest(BaseModel):
    prompt: str

@app.get("/")
def read_root():
    return {"status": "LambadaClips API is running!"}

@app.post("/api/generate")
def generate_image(req: GenerateRequest):
    print(f"\n[API] Menerima prompt: {req.prompt}")
    # Uncomment baris di bawah ini untuk generate gambar sungguhan:
    # image = pipe(prompt=req.prompt, num_inference_steps=25).images[0]
    # buffered = BytesIO()
    # image.save(buffered, format="PNG")
    # img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    # return {"image": img_str}
    return {"message": "Ini adalah response API Simulasi. Uncomment script pada notebook untuk aktivasi SDXL."}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Jalankan API di background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("FastAPI Server berjalan di background...")

In [ ]:
# 3. Expose ke Internet Publik menggunakan Ngrok
# GANTI DENGAN TOKEN NGROK ANDA DARI https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "REPLACE_WITH_YOUR_NGROK_TOKEN_HERE" 

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000).public_url
    print("\n" + "="*60)
    print("✅ CLOUD URL LAMBADACLIPS BERHASIL DIBUAT!")
    print("\n👉 URL ANDA:", public_url)
    print("\nSalin URL di atas dan tempelkan ke form 'SDXL API URL' di Settings LambadaClips.")
    print("="*60)
except Exception as e:
    print("Gagal menghubungkan ke Ngrok. Pastikan token sudah benar.", e)
